# Step 3: Create GitHub Environments

![GitHub](https://img.shields.io/badge/GitHub-181717?logo=github&logoColor=white)
![GitHub CLI](https://img.shields.io/badge/GitHub_CLI-181717?logo=github&logoColor=white)

This notebook creates GitHub environments (dev, staging, production) in each repository and optionally adds production protection rules.

**What this notebook does:**
1. Loads configuration from `config.json` (or collects it fresh)
2. Fetches the repository list
3. Creates environments via the GitHub API
4. Optionally adds required reviewers to the production environment

> **Prerequisite:** Run **[02_setup_credentials.ipynb](02_setup_credentials.ipynb)** first, or at least have `config.json` with your org name and repo list.

> **Personal accounts:** This notebook works for both GitHub organizations and personal accounts. Set your GitHub username as the owner. Note that **environment protection rules** (required reviewers, wait timers) require **GitHub Pro or higher** for private repos — on free personal accounts, environment creation will succeed but protection rules will be skipped gracefully.

## Prerequisites Check

In [ ]:
import subprocess, json, sys, os

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Check GitHub CLI
r = run_cmd("gh --version", check=False)
if r and r.returncode == 0:
    print(f"✓ GitHub CLI installed: {r.stdout.strip().splitlines()[0]}")
else:
    print("✗ GitHub CLI not found. Install: https://cli.github.com")
    raise SystemExit("GitHub CLI is required. Install it and re-run this cell.")

# Check GitHub authentication
r = run_cmd("gh auth status", check=False)
if r and r.returncode == 0:
    print("✓ GitHub CLI is authenticated.")
else:
    print("✗ Not authenticated. Run 'gh auth login' in a terminal first.")
    raise SystemExit("GitHub CLI authentication required. Run 'gh auth login' and re-run this cell.")

print("\n✓ All prerequisites met.")

## Load Configuration

In [ ]:
# Load config from previous notebooks
from pathlib import Path

config_path = Path.cwd() / "config.json"
config = {}
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    print(f"✓ Loaded config from {config_path}")
    # Validate required keys
    required_keys = ["org_name", "environments"]
    missing = [k for k in required_keys if k not in config or not config[k]]
    if missing:
        print(f"⚠ config.json is missing required keys: {missing}")
else:
    print(f"No config.json found at {config_path}")
    print("Tip: Copy config.json.example to config.json and fill in your values, or run notebook 01 first.")

GITHUB_OWNER = config.get("org_name") or input("GitHub owner (org name or personal username): ").strip()
ENVIRONMENTS = config.get("environments", ["dev", "staging", "production"])
REPOS = config.get("repos", [])

if not REPOS:
    mode = input("No repos in config. Fetch from GitHub? (y/n): ").strip().lower()
    if mode == "y":
        r = run_cmd(f"gh repo list {GITHUB_OWNER} --limit 1000 --json name --jq '.[].name'", check=False)
        if r and r.returncode == 0:
            REPOS = [line.strip() for line in r.stdout.strip().split("\n") if line.strip()]
        else:
            print(f"✗ Failed to fetch repos: {r.stderr.strip() if r else ''}")
    else:
        repos_input = input("Enter repo names (comma-separated): ").strip()
        REPOS = [item.strip() for item in repos_input.split(",") if item.strip()]

print(f"\nGitHub owner: {GITHUB_OWNER}")
print(f"Environments: {ENVIRONMENTS}")
print(f"Repositories: {len(REPOS)}")

## Configure Production Protection (Optional)

Optionally set a required reviewer for the production environment. Leave blank to skip.

In [ ]:
# --- Production Protection Rules ---
# Optionally add a required reviewer to the production environment.

PROD_REVIEWER_USER_ID = ""

if "production" not in ENVIRONMENTS:
    print(f"⊘ No 'production' in ENVIRONMENTS ({ENVIRONMENTS}). Skipping protection rules.")
else:
    # Fetch the authenticated user's info
    r = run_cmd("gh api /user --output json", check=False)
    if r and r.returncode == 0:
        try:
            user_data = json.loads(r.stdout)
            gh_username = user_data.get("login", "unknown")
            gh_user_id = str(user_data.get("id", ""))
            print(f"Authenticated as: {gh_username} (ID: {gh_user_id})")
            choice = input(f"Use '{gh_username}' as production reviewer? (y/n/skip): ").strip().lower()
            if choice == "y":
                PROD_REVIEWER_USER_ID = gh_user_id
            elif choice == "n":
                uid = input("Enter reviewer's GitHub numeric user ID: ").strip()
                if uid.isdigit():
                    PROD_REVIEWER_USER_ID = uid
                else:
                    print(f"⚠ '{uid}' is not a valid numeric ID. Skipping protection rules.")
        except json.JSONDecodeError:
            print(f"⚠ Could not parse user info. Raw output: {r.stdout[:200]}")
            uid = input("Enter reviewer's GitHub numeric user ID (blank to skip): ").strip()
            if not uid or uid.isdigit():
                PROD_REVIEWER_USER_ID = uid
            else:
                print(f"⚠ '{uid}' is not a valid numeric ID. Skipping protection rules.")
    else:
        print(f"⚠ Could not fetch user info: {r.stderr.strip() if r else 'gh api failed'}")
        uid = input("Enter reviewer's GitHub numeric user ID (blank to skip): ").strip()
        if not uid or uid.isdigit():
            PROD_REVIEWER_USER_ID = uid
        else:
            print(f"⚠ '{uid}' is not a valid numeric ID. Skipping protection rules.")

    if PROD_REVIEWER_USER_ID:
        print(f"\n✓ Production reviewer: user ID {PROD_REVIEWER_USER_ID}")
    else:
        print("\n⊘ Skipping production protection rules.")

## Create Environments

Creates dev, staging, and production environments in each repository via the GitHub API.

In [ ]:
# --- Create GitHub Environments ---

import tempfile

results = []

for repo in REPOS:
    print(f"\nSetting up environments for {repo}...")

    for env in ENVIRONMENTS:
        cmd = (
            f'gh api --method PUT '
            f'-H "Accept: application/vnd.github+json" '
            f'"/repos/{GITHUB_OWNER}/{repo}/environments/{env}" '
            f'-F "wait_timer=0"'
        )
        r = run_cmd(cmd, check=False)
        output = (r.stdout + r.stderr) if r else ""

        if r and r.returncode == 0:
            status = "✓ Created"
            print(f"  ✓ Created environment: {env}")
        else:
            status = "✗ Failed"
            print(f"  ✗ Failed: {env}")
            print(f"    {output.strip()[:200]}")

        results.append({"repo": repo, "env": env, "status": status})

        # Add production protection rules
        if env == "production" and PROD_REVIEWER_USER_ID:
            body = {
                "prevent_self_review": True,
                "reviewers": [{"type": "User", "id": int(PROD_REVIEWER_USER_ID)}]
            }
            with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tf:
                json.dump(body, tf)
                tf_path = tf.name
            try:
                prot_cmd = (
                    f'gh api --method PUT '
                    f'-H "Accept: application/vnd.github+json" '
                    f'"/repos/{GITHUB_OWNER}/{repo}/environments/production" '
                    f'--input "{tf_path}"'
                )
                pr = run_cmd(prot_cmd, check=False)
            finally:
                os.unlink(tf_path)
            if pr and pr.returncode == 0:
                print(f"  ✓ Added protection rules for production")
            else:
                prot_output = (pr.stdout + pr.stderr) if pr else ""
                print(f"  ⚠ Failed to add protection rules")
                print(f"    (Protection rules require GitHub Pro/Team/Enterprise for private repos)")
                print(f"    {prot_output.strip()[:200]}")

## Results Table

In [ ]:
# --- Display results ---
from IPython.display import display, HTML

if results:
    rows = "".join(
        f"<tr><td>{r['repo']}</td><td>{r['env']}</td><td>{r['status']}</td></tr>"
        for r in results
    )
    display(HTML(
        "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse'>"
        "<tr><th>Repo</th><th>Environment</th><th>Status</th></tr>"
        f"{rows}</table>"
    ))
else:
    print("No results to display. Run the 'Create Environments' cell first.")

print("\nEnvironment setup complete!")

## Next Steps

Proceed to **[04_configure_secrets_and_workflow.ipynb](04_configure_secrets_and_workflow.ipynb)** to set up GitHub organization secrets and deploy the workflow template.